# Forge3D — Free 3D Pipeline (Colab, $0 forever)

Image → textured GLB → quad remesh → auto-rig → export. No API keys, no paid
services, MIT/Apache-2.0 only.

**Hardware honesty (see docs/HARDWARE_MATRIX.md):**
- Free Colab **T4 (16 GB)**: TripoSR runs great (seconds per model). This is the primary free path.
- TRELLIS.2 (best quality, MIT) officially wants 24 GB VRAM — run that section on Kaggle/Lightning/A100, not here.
- Hunyuan3D 2.1 is **not** in this pipeline (license conflict — see docs/WEIGHTS.md).


In [ ]:
# 1. What GPU did we get?
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Clone Forge3D (free-pipeline branch) + install deps
!git clone --depth 1 --branch feature/forge3d-free-pipeline https://github.com/mhvnsnt/Forge3D.git
%cd Forge3D
!pip install -q -r requirements-local.txt
!python -m forge3d selftest

In [ ]:
# 3. Vendor the official TripoSR repo (MIT) + fetch weights (MIT, ~838MB bf16)
!mkdir -p forge3d/providers/_vendor && git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git forge3d/providers/_vendor/triposr
!mkdir -p ~/.forge3d/weights/triposr
!wget -q -O ~/.forge3d/weights/triposr/config.yaml https://huggingface.co/stabilityai/TripoSR/resolve/main/config.yaml
!wget -q -O ~/.forge3d/weights/triposr/model.ckpt https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt
!ls -lh ~/.forge3d/weights/triposr/

In [ ]:
# 4. Get an input image (upload your own, or grab the sample)
from google.colab import files
try:
    uploaded = files.upload()   # pick a PNG/JPG: front-facing, plain background works best
    IMAGE = list(uploaded.keys())[0]
except Exception:
    IMAGE = None
print("IMAGE =", IMAGE)

In [ ]:
# 5. GENERATE — TripoSR on the free T4 (seconds)
# Full pipeline: generate -> cleanup -> quad remesh -> texture -> rig -> export
!python -m forge3d generate --provider triposr --image "$IMAGE" --out out/triposr_run \
    --quadremesh --quad-target 30000 --rig --wait-for-ram
!ls -lh out/triposr_run/

In [ ]:
# 6. Inspect the run manifest (provenance: provider, stages, timings)
import json
print(open("out/triposr_run/run.json").read())

In [ ]:
# 7. Download your model (GLB opens in Blender, Windows 3D Viewer, etc.)
from google.colab import files
import glob
for f in sorted(glob.glob("out/triposr_run/*.glb")):
    print("downloading", f)
    files.download(f)

---
## Quality pass: TRELLIS.2 (needs 24 GB VRAM — Kaggle / Lightning / A100, NOT free T4)

Run cells 1–2 above on the bigger GPU first, then:


In [ ]:
# 8. TRELLIS.2 setup (Linux + CUDA 12.x only). Takes ~10-20 min (builds CUDA ops).
!git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git
%cd TRELLIS.2
!bash setup.sh
%cd /content/Forge3D
!huggingface-cli download microsoft/TRELLIS.2-4B --local-dir ~/.forge3d/weights/trellis2
# NOTE: DINOv3 weights are HF-gated — accept Meta's terms at
# https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m first.

In [ ]:
# 9. GENERATE with TRELLIS.2 (native PBR GLB, best open quality)
!FORGE3D_TRELLIS2_REPO=/content/TRELLIS.2 python -m forge3d generate \
    --provider trellis2 --image "$IMAGE" --out out/trellis2_run \
    --quadremesh --rig --no-densify
!ls -lh out/trellis2_run/
from google.colab import files
import glob
for f in sorted(glob.glob("out/trellis2_run/*.glb")):
    files.download(f)

---
## Export formats
The pipeline exports GLB by default. For FBX/OBJ, re-export any GLB:
```
!FORGE3D_BLENDER_BIN=/path/to/blender python - <<'EOF'
# (or use the pipeline's export stage — see docs)
EOF
```
GLB → import straight into Blender / Unity / Unreal / Godot. Done. $0 forever.
